In [1]:
import pandas as pd
import openai
import json
import backoff
from tqdm import tqdm

## Load data

In [ ]:
# Load urology annotations
cmb_all = pd.read_csv('../data/urology-related/annotations/cmb_all - cmb_pure_actions1.csv')
cmb_inst_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_all - instrument coverage.csv')
cmb_verb_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_all - action_coverage.csv')
cmb_trgt_cvrg = pd.read_csv('../data/urology-related/annotations/cmb_all - anatomic_coverage.csv')

# Format columns
cmb_inst_cvrg = cmb_inst_cvrg.rename(columns={'Instruments_fix': 'instruments', 'count_fix': 'count'})
cmb_verb_cvrg = cmb_verb_cvrg.rename(columns={'action_fix': 'verb', 'count_fix': 'count'})
cmb_trgt_cvrg = cmb_trgt_cvrg.rename(columns={'tissue_fix': 'target', 'count_fix': 'count'})

In [122]:
cmb_all['action']

0       [Check on the left side]
1                   [tap little]
2                         [NONE]
3                         [NONE]
4                         [NONE]
                  ...           
4205                      [NONE]
4206                      [NONE]
4207                      [NONE]
4208               [come closer]
4209                      [NONE]
Name: action, Length: 4210, dtype: object

## Map Urology IAT to Clusters

In [128]:
from ast import literal_eval
import re

action_clusters = pd.read_csv('../data/urology-related/rafal_action_clusters.csv')
instrument_clusters = pd.read_csv('../data/urology-related/rafal_instrument_clusters.csv')
tissue_clusters = pd.read_csv('../data/urology-related/rafal_tissue_clusters.csv')

# def eval_document(df):
#     df['Document'] = df['Document'].apply(lambda x: None if x is not None and 'NONE' in x else x)
#     df['Document'] = df['Document'].apply(lambda x: re.sub(r'\["*', '["', x) if isinstance(x, str) else x)
#     df['Document'] = df['Document'].apply(lambda x: re.sub(r'"*\]', '"]', x) if isinstance(x, str) else x)
#     df['Document'] = df['Document'].apply(lambda x: x.replace(',', '","') if isinstance(x, str) else x)
#     return df

# action_clusters = eval_document(action_clusters)
# instrument_clusters = eval_document(instrument_clusters)
# tissue_clusters = eval_document(tissue_clusters)

# def eval_strings(df, cols):
#     for col in cols:
#         df[col] = df[col].apply(lambda x: eval(x) if isinstance(x, str) else x)
#     return df

# eval_cols = ['Document', 'Representation', 'Representative_Docs']
# action_clusters = eval_strings(action_clusters, eval_cols)
# instrument_clusters = eval_strings(instrument_clusters, eval_cols)
# tissue_clusters = eval_strings(tissue_clusters, eval_cols)

# print(f"Before expansion: len(action_clusters):       {len(action_clusters)}")
# print(f"Before expansion: len(instrument_clusters):   {len(instrument_clusters)}")
# print(f"Before expansion: len(tissue_clusters):       {len(tissue_clusters)}")

# def expand_document(df):
#     # First make sure Document is properly evaluated as a list
#     df['Document'] = df['Document'].apply(lambda x: x if isinstance(x, list) else [x])

#     # Then explode the Document column
#     expanded_df = df.explode('Document')

#     # Reset the index for the new expanded dataframe
#     expanded_df = expanded_df.reset_index(drop=True)

#     return expanded_df

# action_clusters = expand_document(action_clusters)
# instrument_clusters = expand_document(instrument_clusters)
# tissue_clusters = expand_document(tissue_clusters)

# print(f"After expansion: len(action_clusters):        {len(action_clusters)}")
# print(f"After expansion: len(instrument_clusters):    {len(instrument_clusters)}")
# print(f"After expansion: len(tissue_clusters):        {len(tissue_clusters)}")

In [130]:
action_clusters

,Unnamed: 0,Document,Topic,Name,Representation,Representative_Docs,Top_n_words,Probability,Representative_document
0,47,[sweep off],0,0_sweep_shorter_buzz_sweeping,"['sweep', 'shorter', 'buzz', 'sweeping', 'down...","['[sweep]', '[sweep off]', '[sweep]']",sweep - shorter - buzz - sweeping - down - tow...,0.067351,True
1,57,[sweep],0,0_sweep_shorter_buzz_sweeping,"['sweep', 'shorter', 'buzz', 'sweeping', 'down...","['[sweep]', '[sweep off]', '[sweep]']",sweep - shorter - buzz - sweeping - down - tow...,1.000000,True
2,59,[sweep laterally],0,0_sweep_shorter_buzz_sweeping,"['sweep', 'shorter', 'buzz', 'sweeping', 'down...","['[sweep]', '[sweep off]', '[sweep]']",sweep - shorter - buzz - sweeping - down - tow...,0.189651,False
3,167,[don't sweep],0,0_sweep_shorter_buzz_sweeping,"['sweep', 'shorter', 'buzz', 'sweeping', 'down...","['[sweep]', '[sweep off]', '[sweep]']",sweep - shorter - buzz - sweeping - down - tow...,0.167978,False
4,168,[do shorter sweeps],0,0_sweep_shorter_buzz_sweeping,"['sweep', 'shorter', 'buzz', 'sweeping', 'down...","['[sweep]', '[sweep off]', '[sweep]']",sweep - shorter - buzz - sweeping - down - tow...,0.861203,False
...,...,...,...,...,...,...,...,...,...
1882,1434,"[define, expose]",38,38_show_expose_present_define,"['show', 'expose', 'present', 'define', 'test'...","['[show]', '[show]', '[show]']",show - expose - present - define - test - find...,1.000000,False
1883,1563,[show],38,38_show_expose_present_define,"['show', 'expose', 'present', 'define', 'test'...","['[show]', '[show]', '[show]']",show - expose - present - define - test - find...,1.000000,True
1884,1721,[Present],38,38_show_expose_present_define,"['show', 'expose', 'present', 'define', 'test'...","['[show]', '[show]', '[show]']",show - expose - present - define - test - find...,1.000000,False
1885,1799,[show],38,38_show_expose_present_define,"['show', 'expose', 'present', 'define', 'test'...","['[show]', '[show]', '[show]']",show - expose - present - define - test - find...,1.000000,True


## Get Valid Annotations

In [132]:
instrument_mappings = instrument_clusters.set_index('Document')['Name'].to_dict()
action_mappings = action_clusters.set_index('Document')['Name'].to_dict()
tissue_mappings = tissue_clusters.set_index('Document')['Name'].to_dict()

In [135]:
annotations_df = cmb_all.copy()
annotations_df['instrument-cluster'] = annotations_df['instrument'].map(instrument_mappings)
annotations_df['action-cluster'] = annotations_df['action'].map(action_mappings)
annotations_df['tissue-cluster'] = annotations_df['tissue'].map(tissue_mappings)

annotations_df['instrument-cluster'] = annotations_df['instrument-cluster'].apply(lambda x: x if x != 'NONE' else None)
annotations_df['action-cluster'] = annotations_df['action-cluster'].apply(lambda x: x if x != 'NONE' else None)
annotations_df['tissue-cluster'] = annotations_df['tissue-cluster'].apply(lambda x: x if x != 'NONE' else None)

In [137]:
len(annotations_df.dropna(subset='instrument-cluster')), len(annotations_df.dropna(subset='action-cluster')), len(annotations_df.dropna(subset='tissue-cluster'))

(564, 1897, 702)

In [138]:
annotations_df.dropna(subset='instrument-cluster')['instrument-cluster'].value_counts()

instrument-cluster
0_left_hand_camera_arm                     69
1_electrocautery_energy_devices_device     65
2_needle_needles_sutures_clips             38
3_scissors_driver_scissor_needle           37
4_sucker_instrument_right_hand             35
5_stitch_thinpoint_vicyl_chromic           34
6_4th_arm__                                33
7_fourth_arm_tenaculum_stitch              32
8_retractor_locking_liver_alice            25
9_suture_suction_prograsp_perforator       23
10_clip_foley_clips_weck                   21
11_catheter_sealer_vicryl_coagulator       19
12_graspers_grasper_electrocautery_left    17
14_right_hand__                            17
13_stitches_weck_sutures_robotic           17
16_camera___                               16
15_hook_drain_bovie_stent                  16
17_stapler_spreader_tool_cutting           14
18_sutures_jaws_camera_suture              13
19_retractor_fourth_arm_left               12
20_bipolar_monopolar_weck_                 11
Name: count, dt

In [139]:
annotations_df.dropna(subset='action-cluster')['action-cluster'].value_counts()

action-cluster
1_pull_grab_drop_regrab             122
2_back_travel_bit_little            105
4_take_watch_do_it                   81
0_sweep_shorter_buzz_sweeping        80
6_higher_slow_go_below               75
3_buzz_buzzing_not_frindge           71
5_side_lateral_the_other             67
11_open_up_press_spread              63
7_open_see_apply_completely          58
19_closer_close_let_come             57
14_off_leave_out_right               57
8_grab_swipe_scoop_lower             55
26_clean_follow_burn_coagulate       55
15_through_don_do_strangulate        53
9_cut_cutting_cold_scrape            53
10_stop_wait_short_                  52
23_push_pull_more_tight              51
24_lift_hold_advance_up              51
13_careful_be_rip_gentle             48
12_turn_rotate_flip_round            45
33_retract_switch_bring_down         44
16_going_keep_continue_following     42
18_stay_right_keep_remain            39
17_go_along_anchor_change            39
34_move_wrist_smaller_han

In [140]:
annotations_df.dropna(subset='tissue-cluster')['tissue-cluster'].value_counts()

tissue-cluster
6_prostate_side_apex_apical             85
0_vein_veins_complex_dorsal             74
2_ureter_gonadal_upper_gondal           50
3_tissue_adrenal_gland_node             43
4_vessel_vessels_thicker_nerves         42
5_bladder_catheter_on_needle            39
8_seminal_vesicles_vesicle_medial       33
7_muscle_fibers_detrusor_capsule        33
9_urethra_dvc_edge_                     29
11_mucosa_mucosal_inner_edge            25
12_fat_fascia_abdominal_endopelvic      25
10_structure_lateral_anterior_median    25
13_posterior_urethra_bladder_neck       23
14_artery_cava_sac_iliac                21
15_neck_sphincter_bladder_apex          18
16_kidney_liver_pancreas_               18
18_bowel_rectum_rectal_pedicles         17
17_bone_pubic_pelvic_umbilical          17
20_nerve_zone_peripheral_lower          16
19_deferens_vas_vessel_                 16
22_peritoneum_muscle__                  15
21_adenoma_cyst_tumor_capsule           15
-1_contour_psoas_sheets_pedicle        

In [142]:
annotations_df.to_csv('../data/urology-related/annotations/cmb_all-cluster_mappings.csv')